# 40. HS4 TF-IDF Baseline under Matched Evaluation

이번 실험은 최종 모델과 동일한 평가 모집단 및 검증 구조에서 전통적인 TF-IDF + LinearSVC 기준선을 재평가합니다.

- 동일 평가 모집단: 195건, HS4 5개, 제품군 29개
- Outer 평가: Leave-One-Product-Family-Out (LOPO)
- Inner 튜닝: 반복 StratifiedGroupKFold (5 folds × 5 seeds)
- 텍스트: `model_text`
- 특징: Word TF-IDF + Character TF-IDF
- 분류기: class-balanced LinearSVC
- C 후보: 0.3, 1.0, 3.0
- Inner/Outer Macro-F1: 항상 전체 HS4 5개 클래스 기준
- Google Drive 미사용

이 실험은 TF-IDF 기준선을 현재 평가 체계에 맞춰 산출하는 것이 목적이며, SBERT·해설서 모델은 39번 결과와 동일한 OOF 평가 표본 기준으로 비교합니다.

## 실험 설계

각 Outer Fold에서 보류된 제품군을 평가 데이터로 사용하고, 나머지 제품군만으로 Inner 반복 StratifiedGroupKFold를 구성합니다.

각 Inner split마다 TF-IDF 벡터라이저는 Inner 학습 텍스트에만 적합(fit)하고 검증 텍스트에는 transform만 적용합니다. 따라서 검증 데이터의 어휘 정보가 학습에 유입되지 않습니다.

C 값은 Inner 평균 Macro-F1을 기준으로 선택하며 동점이면 Accuracy를 비교합니다. 선택된 C로 Outer 학습 데이터 전체에 벡터라이저와 LinearSVC를 다시 적합한 뒤 보류 제품군을 예측합니다.

희소 HS4 클래스로 인해 학습 데이터에 5개 클래스가 모두 존재하지 않는 Inner split은 39번과 같은 기준으로 제외하고, seed별 커버리지를 별도로 저장합니다. Macro-F1은 검증 데이터에 일부 클래스가 없더라도 항상 전체 5개 HS4 클래스를 기준으로 계산합니다.

## 0. 환경 설정

In [ ]:
!pip -q install -U scikit-learn scipy
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
import requests
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
WORK=Path('/content/40_hs4_tfidf_matched')
WORK.mkdir(parents=True,exist_ok=True)
BASE='https://raw.githubusercontent.com/lin1118-code/HS-complete-product/feature/explanatory-note-integration/'
print(WORK)

## 1. 실험 29와 동일한 데이터 파이프라인

In [ ]:
src=requests.get(BASE+'29_discriminative_features_experiment.ipynb').json()
for i in [3,5,7]: exec(''.join(src['cells'][i]['source']),globals())
print('평가 표본:',len(eval_df),'| HS6:',len(labels),'| 제품군:',eval_df.product_family.nunique(),'| HS4:',eval_df.hs4.nunique())
assert len(eval_df)==195 and len(labels)==11 and eval_df.product_family.nunique()==29 and eval_df.hs4.nunique()==5
hs4_labels=sorted(eval_df.hs4.astype(str).unique())
print('HS4 labels:',hs4_labels)
display(eval_df.groupby('hs4').agg(n=('hs4','size'),families=('product_family','nunique')))
display(pd.crosstab(eval_df.product_family,eval_df.hs4))

## 2. TF-IDF 모델 및 평가 함수

In [ ]:
C_VALUES=[0.3,1.0,3.0]
INNER_SEEDS=[11,23,37,53,71]
INNER_SPLITS=5
def make_vectorizer():
    return FeatureUnion([('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=1,max_features=30000,sublinear_tf=True)),('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=1,max_features=30000,sublinear_tf=True))])
def fit_predict_tfidf(train_idx,valid_idx,c):
    train=eval_df.iloc[train_idx]
    valid=eval_df.iloc[valid_idx]
    vectorizer=make_vectorizer()
    x_train=vectorizer.fit_transform(train.model_text.astype(str))
    x_valid=vectorizer.transform(valid.model_text.astype(str))
    clf=LinearSVC(C=c,class_weight='balanced',random_state=42)
    clf.fit(x_train,train.hs4.astype(str))
    return clf.predict(x_valid)
def build_inner_splits(outer_train):
    inner_df=eval_df.iloc[outer_train].reset_index(drop=True)
    y=inner_df.hs4.astype(str).to_numpy()
    groups=inner_df.product_family.to_numpy()
    all_labels=set(hs4_labels)
    repeats=[]
    audits=[]
    for seed in INNER_SEEDS:
        splitter=StratifiedGroupKFold(n_splits=INNER_SPLITS,shuffle=True,random_state=seed)
        valid_splits=[]
        validated=set()
        skipped=[]
        for split_no,(itr,iva) in enumerate(splitter.split(inner_df,y,groups),1):
            global_tr=outer_train[itr]
            global_va=outer_train[iva]
            train_labels=set(eval_df.iloc[global_tr].hs4.astype(str))
            if train_labels!=all_labels:
                skipped.append('|'.join(sorted(set(inner_df.iloc[iva].product_family.astype(str)))))
                continue
            valid_splits.append((global_tr,global_va))
            validated.update(map(int,global_va))
        if not valid_splits: raise ValueError(f'Inner seed {seed}: 유효 split이 없습니다.')
        excluded=sorted(set(map(int,outer_train))-validated)
        audits.append({'seed':seed,'valid_splits':len(valid_splits),'validation_rows':len(validated),'coverage_rate':len(validated)/len(outer_train),'excluded_families':'|'.join(sorted(set(eval_df.iloc[excluded].product_family.astype(str)))) if excluded else '','skipped_split_groups':' ; '.join(skipped)})
        repeats.append(valid_splits)
    return repeats,audits
def score_c(repeats,c):
    f1s=[]; accs=[]; row_counts=[]; class_counts=[]
    for split_list in repeats:
        yt=[]; yp=[]
        for train_idx,valid_idx in split_list:
            pred=fit_predict_tfidf(train_idx,valid_idx,c)
            yt.extend(eval_df.iloc[valid_idx].hs4.astype(str).tolist())
            yp.extend(pred.tolist())
        f1s.append(f1_score(yt,yp,labels=hs4_labels,average='macro',zero_division=0))
        accs.append(accuracy_score(yt,yp))
        row_counts.append(len(yt))
        class_counts.append(len(set(yt)))
    return float(np.mean(f1s)),float(np.mean(accs)),float(np.std(f1s,ddof=1)),float(np.mean(row_counts)),float(np.mean(class_counts))
def tune_c(repeats):
    rows=[]
    for c in C_VALUES:
        mf,acc,sd,nrows,nclasses=score_c(repeats,c)
        rows.append({'C':c,'mean_inner_macro_f1':mf,'mean_inner_accuracy':acc,'std_inner_macro_f1':sd,'mean_validation_rows_per_seed':nrows,'mean_validated_classes':nclasses})
    table=pd.DataFrame(rows).sort_values(['mean_inner_macro_f1','mean_inner_accuracy'],ascending=False).reset_index(drop=True)
    return float(table.iloc[0].C),table
splits=list(LeaveOneGroupOut().split(eval_df,eval_df.hs4,groups=eval_df.product_family))
print('Outer LOPO:',len(splits),'| C candidates:',C_VALUES,'| Inner:',len(INNER_SEEDS),'seeds ×',INNER_SPLITS,'folds')

## 3. Outer LOPO 실행 (5개 Fold씩)

In [ ]:
FOLD_START=1
FOLD_END=5
def fold_path(f): return WORK/f'fold_{f:02d}.csv'
def tuning_path(f): return WORK/f'fold_{f:02d}_tuning.csv'
def coverage_path(f): return WORK/f'fold_{f:02d}_coverage.csv'
def grid_path(f): return WORK/f'fold_{f:02d}_gridsearch.csv'
for fold,(tr,te) in enumerate(splits,1):
    if not FOLD_START<=fold<=FOLD_END: continue
    if all(p.exists() for p in [fold_path(fold),tuning_path(fold),coverage_path(fold),grid_path(fold)]):
        print('Fold',fold,'완료 - skip')
        continue
    if set(eval_df.iloc[tr].hs4.astype(str))!=set(hs4_labels): raise ValueError(f'Fold {fold}: Outer 학습 라벨 누락')
    repeats,coverage=build_inner_splits(tr)
    selected_c,grid=tune_c(repeats)
    pred=fit_predict_tfidf(tr,te,selected_c)
    out=pd.DataFrame({'row_index':te.astype(int),'fold':fold,'heldout_family':eval_df.iloc[te].product_family.to_numpy(),'true_hs4':eval_df.iloc[te].hs4.astype(str).to_numpy(),'model':'TFIDF_LinearSVC_Tuned','predicted_hs4':pred})
    out.to_csv(fold_path(fold),index=False,encoding='utf-8-sig')
    pd.DataFrame([{'fold':fold,'selected_C':selected_c,'mean_inner_macro_f1':grid.iloc[0].mean_inner_macro_f1,'mean_inner_accuracy':grid.iloc[0].mean_inner_accuracy,'std_inner_macro_f1':grid.iloc[0].std_inner_macro_f1}]).to_csv(tuning_path(fold),index=False,encoding='utf-8-sig')
    grid.assign(outer_fold=fold).to_csv(grid_path(fold),index=False,encoding='utf-8-sig')
    pd.DataFrame(coverage).assign(outer_fold=fold).to_csv(coverage_path(fold),index=False,encoding='utf-8-sig')
    print(f'Fold {fold}/{len(splits)} | C={selected_c} | 평균 Inner 커버리지={np.mean([x["coverage_rate"] for x in coverage]):.3f}')
print('현재 실행 구간 완료')

## 4. 진행률 및 중간 백업

In [ ]:
done=sorted(int(p.stem.split('_')[1]) for p in WORK.glob('fold_[0-9][0-9].csv'))
missing=sorted(set(range(1,len(splits)+1))-set(done))
print(f'완료 Fold: {len(done)}/{len(splits)} | 누락: {missing}')
BACKUP='/content/40_fold_backup.zip'
with zipfile.ZipFile(BACKUP,'w',zipfile.ZIP_DEFLATED) as z:
    for pattern in ['fold_[0-9][0-9].csv','fold_[0-9][0-9]_tuning.csv','fold_[0-9][0-9]_coverage.csv','fold_[0-9][0-9]_gridsearch.csv']:
        for p in WORK.glob(pattern): z.write(p,p.name)
from google.colab import files
files.download(BACKUP)

## 5. 중간 백업 복구 (새 런타임에서만 실행)

In [ ]:
from google.colab import files
uploaded=files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z: z.extractall(WORK)
        print('복구 완료:',name)

## 6. 통합 평가 및 최종 결과 ZIP

In [ ]:
files_list=sorted(WORK.glob('fold_[0-9][0-9].csv'))
done=sorted(int(p.stem.split('_')[1]) for p in files_list)
missing=sorted(set(range(1,len(splits)+1))-set(done))
print(f'완료 Fold: {len(done)}/{len(splits)} | 누락: {missing}')
if missing: raise ValueError('미완료 Outer Fold가 있습니다.')
pred=pd.concat([pd.read_csv(p) for p in files_list],ignore_index=True)
if pred.duplicated(['row_index']).any(): raise ValueError('중복 예측 존재')
yt=pred.true_hs4.astype(str); yp=pred.predicted_hs4.astype(str)
summary=pd.DataFrame([{'model':'TFIDF_LinearSVC_Tuned','n':len(pred),'HS4_Macro_F1':f1_score(yt,yp,labels=hs4_labels,average='macro',zero_division=0),'HS4_Accuracy':accuracy_score(yt,yp)}])
p,r,f,s=precision_recall_fscore_support(yt,yp,labels=hs4_labels,zero_division=0)
class_metrics=pd.DataFrame([{'HS4':c,'support':int(sv),'precision':pv,'recall':rv,'F1':fv} for c,pv,rv,fv,sv in zip(hs4_labels,p,r,f,s)])
tuning=pd.concat([pd.read_csv(p) for p in sorted(WORK.glob('fold_[0-9][0-9]_tuning.csv'))],ignore_index=True)
coverage=pd.concat([pd.read_csv(p) for p in sorted(WORK.glob('fold_[0-9][0-9]_coverage.csv'))],ignore_index=True)
gridsearch=pd.concat([pd.read_csv(p) for p in sorted(WORK.glob('fold_[0-9][0-9]_gridsearch.csv'))],ignore_index=True)
display(summary.round(4))
display(class_metrics.round(4))
display(tuning)
display(coverage.sort_values(['outer_fold','seed']))
display(gridsearch.sort_values(['outer_fold','C']))
OUT=WORK/'final'; OUT.mkdir(exist_ok=True)
pred.to_csv(OUT/'40_tfidf_oof_predictions.csv',index=False,encoding='utf-8-sig')
summary.to_csv(OUT/'40_tfidf_model_comparison.csv',index=False,encoding='utf-8-sig')
class_metrics.to_csv(OUT/'40_tfidf_hs4_class_metrics.csv',index=False,encoding='utf-8-sig')
tuning.to_csv(OUT/'40_tfidf_selected_C_by_fold.csv',index=False,encoding='utf-8-sig')
coverage.to_csv(OUT/'40_tfidf_inner_validation_coverage.csv',index=False,encoding='utf-8-sig')
gridsearch.to_csv(OUT/'40_tfidf_inner_gridsearch_scores.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(confusion_matrix(yt,yp,labels=hs4_labels),index=hs4_labels,columns=hs4_labels).to_csv(OUT/'40_tfidf_confusion_matrix.csv',encoding='utf-8-sig')
with zipfile.ZipFile('/content/40_hs4_tfidf_matched_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob('*.csv'): z.write(p,p.name)
print('저장 완료: /content/40_hs4_tfidf_matched_results.zip')

## 결과 해석 기준

1. TF-IDF 기준선은 39번과 동일한 195건, 5개 HS4, 29개 제품군의 Outer LOPO OOF 예측으로 평가합니다.
2. C는 각 Outer Fold의 Inner 검증에서 선택하므로 Outer 평가 데이터가 하이퍼파라미터 선택에 사용되지 않습니다.
3. TF-IDF 벡터라이저는 매 Inner split의 학습 데이터에만 적합합니다.
4. 최종 비교에서는 이 실험의 OOF 예측과 39번의 OOF 예측을 사용합니다. 서로 다른 평가 모집단이나 분할 방식의 과거 결과와 직접 비교하지 않습니다.
5. TF-IDF의 성능이 낮더라도 이는 단어·문자 기반 선형 분류기의 기준선이며, 모델 우열은 Macro-F1, Accuracy, 클래스별 Recall을 함께 보고 해석합니다.